# EDA: Census Demographics — Population, Age & Race

**Source**: `data/raw/Census-Demographics.csv`  
**Branch**: `002-per-dataset-eda-notebooks`  
**Sections**: 1 Schema | 2 Quality | 3 Distributions | 4 Domain Analysis

In [ ]:
import warnings
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 100
sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

# Locate project root — works from repo root (nbmake) or notebooks/eda/ (Jupyter)
_here = Path.cwd()
_candidates = [_here, _here.parent, _here.parents[1], _here.parents[2]]
PROJECT_ROOT = next((p for p in _candidates if (p / "data" / "raw").exists()), None)
assert PROJECT_ROOT is not None, f"Cannot locate project root from {_here}"
RAW_DATA = PROJECT_ROOT / "data" / "raw"

## Data Loading & Schema Summary


In [ ]:
FILE_NAME = "Census-Demographics.csv"
DATASET_TITLE = "Census Demographics — Population, Age & Race"

df = pd.read_csv(RAW_DATA / FILE_NAME)
print(f"File  : {FILE_NAME}")
print(f"Shape : {df.shape[0]} rows × {df.shape[1]} columns")

In [ ]:
schema = (
    pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "null_count": df.isnull().sum(),
            "null_pct": (df.isnull().mean() * 100).round(2),
            "unique_count": df.nunique(),
        }
    )
    .rename_axis("column")
    .reset_index()
)

display(schema)

In [ ]:
ZIP_COL = "geoid"
if ZIP_COL in df.columns:
    zip_dtype = str(df[ZIP_COL].dtype)
    n_unique = df[ZIP_COL].nunique()
    samples = df[ZIP_COL].dropna().head(5).tolist()
    print(f"✓ ZIP key '{ZIP_COL}' found | dtype: {zip_dtype} | unique ZIPs: {n_unique}")
    print(f"  sample values: {samples}")
    if df[ZIP_COL].dtype == object:
        print("  ⚠ ZIP dtype is string — verify 5-digit format consistency")
else:
    print(f"⚠ ZIP key '{ZIP_COL}' NOT present in this dataset")

## Data Quality Assessment

**Flags are observational warnings only.**
Drop and imputation decisions belong in `src/cleaning.py`, not in notebooks.

In [ ]:
# ── Duplicate rows ───────────────────────────────────────────────────────────
n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

# ── Quality flag assignment ───────────────────────────────────────────────────
ADMIN_COLS = {"feature id", "feature label", "shid", "geoid"}


def assign_flag(null_pct: float, col_dtype: str, col_name: str) -> str:
    """Assign an observational Data Quality Flag.

    Contract: contracts/data-quality-flag-schema.md v1.0

    Args:
        null_pct: Percentage of null values in the column (0-100).
        col_dtype: pandas dtype string (e.g. "float64", "object").
        col_name: Column name; admin columns are always marked usable.

    Returns:
        One of: 'usable', 'needs_cleaning', 'too_sparse'.
    """
    if col_name in ADMIN_COLS:
        return "usable"
    if null_pct > 30.0:
        return "too_sparse"
    if col_dtype == "object":
        return "needs_cleaning"
    return "usable"


quality = (
    pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "null_count": df.isnull().sum(),
            "null_pct": (df.isnull().mean() * 100).round(2),
        }
    )
    .rename_axis("column")
    .reset_index()
)

quality["quality_flag"] = quality.apply(
    lambda row: assign_flag(row["null_pct"], row["dtype"], row["column"]),
    axis=1,
)

print("Data Quality Flags  (usable | needs_cleaning | too_sparse):")
display(quality[["column", "null_pct", "dtype", "quality_flag"]])

counts = quality["quality_flag"].value_counts().to_dict()
print("")
for flag in ("usable", "needs_cleaning", "too_sparse"):
    n = counts.get(flag, 0)
    icon = "✓  " if flag == "usable" else "⚠  "
    print(f"  [{icon}]  {flag}: {n} column(s)")

In [ ]:
# ── IQR outlier detection ─────────────────────────────────────────────────────
# Method: lower = Q1 - 1.5×IQR,  upper = Q3 + 1.5×IQR  (research.md Decision 1)
# Outlier flags are informational only — extreme ZIPs may be analytically important.

ADMIN_COLS = {"feature id", "feature label", "shid", "geoid"}
numeric_domain = [
    c for c in df.select_dtypes(include="number").columns if c not in ADMIN_COLS
]

outlier_rows = []
for col in numeric_domain:
    series = df[col].dropna()
    if series.empty:
        continue
    q1 = float(series.quantile(0.25))
    q3 = float(series.quantile(0.75))
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outlier_mask = (df[col] < lower_fence) | (df[col] > upper_fence)
    outlier_count = int(outlier_mask.sum())
    outlier_zips: list = []
    if "geoid" in df.columns and outlier_count > 0:
        outlier_zips = df.loc[outlier_mask, "geoid"].dropna().astype(str).tolist()[:10]
    outlier_rows.append(
        {
            "column": col,
            "q1": round(q1, 4),
            "q3": round(q3, 4),
            "iqr": round(iqr, 4),
            "lower_fence": round(lower_fence, 4),
            "upper_fence": round(upper_fence, 4),
            "outlier_count": outlier_count,
            "outlier_zips": outlier_zips,
        }
    )

outlier_summary = pd.DataFrame(outlier_rows)
n_with_outliers = int((outlier_summary["outlier_count"] > 0).sum())
print(f"IQR outlier summary  ({len(outlier_summary)} numeric domain columns)")
print(f"Columns with ≥1 outlier: {n_with_outliers}")
display(outlier_summary)

## Univariate Distributions


In [ ]:
ADMIN_COLS = {"feature id", "feature label", "shid", "geoid"}
domain_cols = [c for c in df.columns if c not in ADMIN_COLS]
numeric_cols = df[domain_cols].select_dtypes(include="number").columns.tolist()
cat_cols = [
    c
    for c in df[domain_cols].select_dtypes(exclude="number").columns
    if df[c].nunique() <= 20
]
print(f"Domain columns : {len(domain_cols)}")
print(f"Numeric        : {len(numeric_cols)}")
print(f"Categorical (≤20 unique) : {len(cat_cols)}")

In [ ]:
if numeric_cols:
    n_grid_cols = 3
    n_grid_rows = (len(numeric_cols) + n_grid_cols - 1) // n_grid_cols
    fig, axes = plt.subplots(
        n_grid_rows,
        n_grid_cols,
        figsize=(6 * n_grid_cols, 4 * n_grid_rows),
        squeeze=False,
    )
    axes_flat = axes.flatten()

    for i, col in enumerate(numeric_cols):
        ax = axes_flat[i]
        data = df[col].dropna()
        ax.hist(
            data, bins=20, edgecolor="white", linewidth=0.4, color="#4472C4", alpha=0.85
        )
        label = col if len(col) <= 48 else col[:45] + "..."
        ax.set_title(label, fontsize=8, pad=4)
        ax.set_xlabel("Value", fontsize=8)
        ax.set_ylabel("Count", fontsize=8)
        ax.tick_params(labelsize=7)

    for j in range(len(numeric_cols), len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f"{DATASET_TITLE} — Numeric Column Distributions", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print("No numeric domain columns found.")

In [ ]:
if cat_cols:
    for col in cat_cols:
        freq = df[col].value_counts(dropna=False).reset_index()
        freq.columns = [col, "count"]
        freq["pct"] = (freq["count"] / len(df) * 100).round(2)
        print(f"\n-- {col} ({df[col].nunique()} unique values) --")
        display(freq)
else:
    print("No categorical domain columns with 20 or fewer unique values found.")

## Domain-Relevant Analysis


In [ ]:
METRIC_COL = "Total Population (2020-2024)"
METRIC_LABEL = "Total Population (2020–2024, count)"
RELEVANCE_NOTE = "No geographic area column is present in this file, so population density cannot be computed directly; total population is used as a proxy. Higher-population ZIPs with resource gaps represent the largest communities of unmet need at scale — a critical demand-side weight in the Desert Score."
N = 10

ranked = (
    df[["geoid", METRIC_COL]]
    .dropna(subset=[METRIC_COL])
    .sort_values(METRIC_COL, ascending=False)
    .reset_index(drop=True)
)
top_n = ranked.head(N).copy()
bottom_n = ranked.tail(N).sort_values(METRIC_COL).copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, subset, color, direction in [
    (axes[0], top_n, "#D9534F", "Highest"),
    (axes[1], bottom_n, "#5CB85C", "Lowest"),
]:
    ax.barh(
        subset["geoid"].astype(str),
        subset[METRIC_COL],
        color=color,
        alpha=0.85,
        edgecolor="white",
        linewidth=0.4,
    )
    ax.set_xlabel(METRIC_LABEL, fontsize=10)
    ax.set_ylabel("ZIP Code", fontsize=10)
    ax.set_title(f"{direction} {N} ZIPs  —  {METRIC_LABEL}", fontsize=10, pad=6)
    ax.invert_yaxis()
    ax.tick_params(labelsize=9)

fig.suptitle(f"{DATASET_TITLE}  —  {METRIC_LABEL}", fontsize=12)
plt.tight_layout()
plt.show()

print(f"\nTop {N} ZIPs (highest {METRIC_LABEL}):")
display(top_n)
print(f"\nBottom {N} ZIPs (lowest {METRIC_LABEL}):")
display(bottom_n)

print("\n" + "─" * 60)
print("Resource Desert Relevance:")
print(RELEVANCE_NOTE)